In [0]:
roll_number = "25280091" 
catalog_name = f"{roll_number}-pa2"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.gold")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog_name}`.bronze.temp")

DataFrame[]

In [0]:
volume_path = f"/Volumes/{catalog_name}/bronze/temp"

flight_landing_zone = f"{volume_path}/flight_landing_zone"
flight_landing_zone_part2 = f"{volume_path}/flight_landing_zone_part2"
cdc_zone_part2 = f"{volume_path}/cdc_zone_part2"
flight_reference_data = f"{volume_path}/flight_reference_data"
checkpoints_path = f"{volume_path}/checkpoints"
schemas_path = f"{volume_path}/schemas"

dbutils.fs.mkdirs(flight_landing_zone)
dbutils.fs.mkdirs(flight_landing_zone_part2)
dbutils.fs.mkdirs(cdc_zone_part2)
dbutils.fs.mkdirs(flight_reference_data)
dbutils.fs.mkdirs(checkpoints_path)
dbutils.fs.mkdirs(schemas_path)

True

In [0]:
dbutils.fs.cp(
    "dbfs:/databricks-datasets/flights/departuredelays.csv",
    f"{flight_landing_zone}/departuredelays.csv"
)

dbutils.fs.cp(
    "dbfs:/databricks-datasets/flights/airport-codes-na.txt",
    f"{flight_reference_data}/airport-codes-na.txt"
)

True

In [0]:
from pyspark.sql.functions import current_timestamp, date_format

bronze_schema_path = f"{schemas_path}/bronze_flights_schema"
bronze_checkpoint = f"{checkpoints_path}/bronze_flights_cp"

bronze_df = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "csv")
         .option("cloudFiles.inferColumnTypes", "true")
         .option("cloudFiles.schemaLocation", bronze_schema_path)
         .option("cloudFiles.schemaEvolutionMode", "rescue")
         .option("header", "true")
         .load(flight_landing_zone)
         .withColumn(
             "ingestion_timestamp",
             date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss")
         )
)

bronze_query = (
    bronze_df.writeStream
             .format("delta")
             .outputMode("append")
             .option("checkpointLocation", bronze_checkpoint)
             .trigger(availableNow=True)
             .toTable(f"`{catalog_name}`.bronze.flights")
)

bronze_query.awaitTermination()

Q1: What is `schemaEvolutionMode` used for?

`schemaEvolutionMode` controls how Auto Loader handles new or unexpected columns in incoming files.  
Using `rescue` is appropriate when raw files may evolve and you do not want the stream to fail; unexpected fields are placed into rescued data instead of breaking ingestion.

 Q2: Trigger modes

Common trigger modes:
- **default trigger**: processes data continuously in micro-batches as data arrives
- **processingTime="..."**: runs micro-batches on a fixed schedule
- **availableNow=True**: processes all currently available data and then stops
- **once=True**: older one-shot style trigger for a single pass


In [0]:
from pyspark.sql.functions import col, concat_ws, lit, when, least, greatest, to_timestamp, broadcast

airport_ref_raw = (
    spark.read
         .option("header", "true")
         .option("sep", "\t")
         .csv(flight_reference_data)
)

airport_ref = (
    airport_ref_raw
    .select(
        col("IATA").alias("iata"),
        col("City").alias("city"),
        col("State").alias("state")
    )
    .filter(col("iata").isNotNull())
)

In [0]:
from pyspark.sql.functions import (
    col, concat, lit, when, least, greatest, concat_ws,
    broadcast, lpad, expr
)

bronze_stream = spark.readStream.table(f"`{catalog_name}`.bronze.flights")

typed_df = (
    bronze_stream
    .withColumn("delay", col("delay").cast("int"))
    .withColumn("distance", col("distance").cast("int"))
    .withColumn("origin", col("origin").cast("string"))
    .withColumn("destination", col("destination").cast("string"))
    .withColumn("date", col("date").cast("string"))
)

silver_base = (
    typed_df
    .withColumn("date_padded", lpad(col("date"), 8, "0"))
    .withColumn(
        "event_timestamp",
        expr("try_to_timestamp(concat('2025', date_padded), 'yyyyMMddHHmm')")
    )
)

In [0]:
silver_enriched = (
    silver_base
    .join(
        broadcast(origin_ref),
        silver_base.origin == origin_ref.origin_iata,
        "left"
    )
    .join(
        broadcast(dest_ref),
        silver_base.destination == dest_ref.dest_iata,
        "left"
    )
    .withColumn("departure", concat_ws(", ", col("origin_city"), col("origin_state")))
    .withColumn("arrival", concat_ws(", ", col("dest_city"), col("dest_state")))
    .filter(col("departure") != "")
    .filter(col("arrival") != "")
    .filter(col("event_timestamp").isNotNull())
    .withColumn(
        "take_off_status",
        when(col("delay") < 0, "Early")
        .when((col("delay") >= 0) & (col("delay") < 10), "On Time")
        .when((col("delay") >= 10) & (col("delay") < 30), "Late")
        .otherwise("Delay")
    )
    .withColumn(
        "standardized_route",
        concat_ws("-", least(col("origin"), col("destination")), greatest(col("origin"), col("destination")))
    )
    .withColumn("seq", lit(3))
    .select(
        "date",
        "departure",
        "arrival",
        "standardized_route",
        "distance",
        "delay",
        "take_off_status",
        "ingestion_timestamp",
        "event_timestamp",
        "seq"
    )
)

In [0]:
silver_query = (
    silver_enriched.writeStream
                   .format("delta")
                   .outputMode("append")
                   .option("checkpointLocation", silver_checkpoint)
                   .trigger(availableNow=True)
                   .toTable(f"`{catalog_name}`.silver.flights")
)

silver_query.awaitTermination()

###  How does a broadcast join optimize performance?

A broadcast join sends the small reference table to all executors, so Spark avoids shuffling the large streaming dataset across the cluster. This reduces network cost and makes joins against small lookup tables much faster.

In [0]:
from pyspark.sql.functions import window, avg, sum as _sum, max as _max, round as _round, date_format, expr, count

silver_stream = (
    spark.readStream
         .table(f"`{catalog_name}`.silver.flights")
         .withWatermark("event_timestamp", "7 days")
)

In [0]:
from pyspark.sql.functions import col, avg, round as _round, date_trunc, date_format, add_months

monthly_delays_df = (
    silver_stream
    .filter(col("event_timestamp").isNotNull())
    .groupBy(date_trunc("month", col("event_timestamp")).alias("month_start"))
    .agg(_round(avg(col("delay") / 60.0), 2).alias("moving_avg_delay"))
    .select(
        date_format(col("month_start"), "yyyy-MM-dd HH:mm:ss").alias("window_start"),
        date_format(add_months(col("month_start"), 1), "yyyy-MM-dd HH:mm:ss").alias("window_end"),
        col("moving_avg_delay")
    )
)

monthly_delays_query = (
    monthly_delays_df.writeStream
                     .format("delta")
                     .outputMode("complete")
                     .option("checkpointLocation", f"{checkpoints_path}/gold_monthly_delays_cp")
                     .trigger(availableNow=True)
                     .toTable(f"`{catalog_name}`.gold.monthly_delays")
)

monthly_delays_query.awaitTermination()

In [0]:
biweekly_distance_df = (
    silver_stream
    .groupBy(window(col("event_timestamp"), "14 days"))
    .agg(_round(_sum("distance"), 0).alias("total_distance"))
    .select(
        date_format(col("window.start"), "yyyy-MM-dd HH:mm:ss").alias("window_start"),
        date_format(col("window.end"), "yyyy-MM-dd HH:mm:ss").alias("window_end"),
        col("total_distance")
    )
)

biweekly_distance_query = (
    biweekly_distance_df.writeStream
                        .format("delta")
                        .outputMode("complete")
                        .option("checkpointLocation", f"{checkpoints_path}/gold_biweekly_distance_cp")
                        .trigger(availableNow=True)
                        .toTable(f"`{catalog_name}`.gold.biweekly_distance")
)

biweekly_distance_query.awaitTermination()

In [0]:
from pyspark.sql.functions import (
    col, date_trunc, date_format, add_months,
    max_by, max as _max, round as _round
)

monthly_max_delay_df = (
    silver_stream
    .filter(col("event_timestamp").isNotNull())
    .groupBy(date_trunc("month", col("event_timestamp")).alias("month_start"))
    .agg(
        max_by(col("standardized_route"), col("delay")).alias("worst_route"),
        _round((_max("delay") / 60.0), 2).alias("max_delay")
    )
    .select(
        date_format(col("month_start"), "yyyy-MM-dd HH:mm:ss").alias("window_start"),
        date_format(add_months(col("month_start"), 1), "yyyy-MM-dd HH:mm:ss").alias("window_end"),
        col("worst_route"),
        col("max_delay")
    )
)

monthly_max_delay_query = (
    monthly_max_delay_df.writeStream
                        .format("delta")
                        .outputMode("complete")
                        .option("checkpointLocation", f"{checkpoints_path}/gold_monthly_max_delay_cp")
                        .trigger(availableNow=True)
                        .toTable(f"`{catalog_name}`.gold.monthly_max_delay")
)

monthly_max_delay_query.awaitTermination()

In [0]:
hourly_flight_counts_df = (
    silver_stream
    .groupBy(window(col("event_timestamp"), "1 hour"))
    .agg(count("*").alias("flight_count"))
    .select(
        date_format(col("window.start"), "yyyy-MM-dd HH:mm:ss").alias("window_start"),
        date_format(col("window.end"), "yyyy-MM-dd HH:mm:ss").alias("window_end"),
        col("flight_count")
    )
)

hourly_flight_counts_query = (
    hourly_flight_counts_df.writeStream
                           .format("delta")
                           .outputMode("complete")
                           .option("checkpointLocation", f"{checkpoints_path}/gold_hourly_flight_counts_cp")
                           .trigger(availableNow=True)
                           .toTable(f"`{catalog_name}`.gold.hourly_flight_counts")
)

hourly_flight_counts_query.awaitTermination()

In [0]:
route_sliding_avg_delay_df = (
    silver_stream
    .groupBy(
        window(col("event_timestamp"), "7 days", "1 day"),
        col("standardized_route")
    )
    .agg(_round(avg(col("delay") / 60.0), 2).alias("avg_delay_hours"))
    .select(
        date_format(col("window.start"), "yyyy-MM-dd HH:mm:ss").alias("window_start"),
        date_format(col("window.end"), "yyyy-MM-dd HH:mm:ss").alias("window_end"),
        "standardized_route",
        "avg_delay_hours"
    )
)

route_sliding_avg_delay_query = (
    route_sliding_avg_delay_df.writeStream
                              .format("delta")
                              .outputMode("complete")
                              .option("checkpointLocation", f"{checkpoints_path}/gold_route_sliding_avg_delay_cp")
                              .trigger(availableNow=True)
                              .toTable(f"`{catalog_name}`.gold.route_sliding_avg_delay")
)

route_sliding_avg_delay_query.awaitTermination()

In [0]:
status_sliding_counts_df = (
    silver_stream
    .groupBy(
        window(col("event_timestamp"), "3 days", "12 hours"),
        col("take_off_status")
    )
    .agg(count("*").alias("status_flight_count"))
    .select(
        date_format(col("window.start"), "yyyy-MM-dd HH:mm:ss").alias("window_start"),
        date_format(col("window.end"), "yyyy-MM-dd HH:mm:ss").alias("window_end"),
        "take_off_status",
        "status_flight_count"
    )
)

status_sliding_counts_query = (
    status_sliding_counts_df.writeStream
                            .format("delta")
                            .outputMode("complete")
                            .option("checkpointLocation", f"{checkpoints_path}/gold_status_sliding_counts_cp")
                            .trigger(availableNow=True)
                            .toTable(f"`{catalog_name}`.gold.status_sliding_counts")
)

status_sliding_counts_query.awaitTermination()

In [0]:
weekly_route_distance_df = (
    silver_stream
    .groupBy(
        window(col("event_timestamp"), "7 days"),
        col("standardized_route")
    )
    .agg(_round(_sum("distance"), 0).alias("weekly_route_distance"))
    .select(
        date_format(col("window.start"), "yyyy-MM-dd HH:mm:ss").alias("window_start"),
        date_format(col("window.end"), "yyyy-MM-dd HH:mm:ss").alias("window_end"),
        "standardized_route",
        "weekly_route_distance"
    )
)

weekly_route_distance_query = (
    weekly_route_distance_df.writeStream
                            .format("delta")
                            .outputMode("complete")
                            .option("checkpointLocation", f"{checkpoints_path}/gold_weekly_route_distance_cp")
                            .trigger(availableNow=True)
                            .toTable(f"`{catalog_name}`.gold.weekly_route_distance")
)

weekly_route_distance_query.awaitTermination()

In [0]:
spark.sql("SHOW CATALOGS").display()
spark.sql(f"SHOW SCHEMAS IN `{catalog_name}`").display()
spark.sql(f"SHOW TABLES IN `{catalog_name}`.bronze").display()
spark.sql(f"SHOW TABLES IN `{catalog_name}`.silver").display()
spark.sql(f"SHOW TABLES IN `{catalog_name}`.gold").display()

catalog
25280091-pa2
25280091_pa1
main
samples
system
workspace


databaseName
bronze
default
gold
information_schema
silver


database,tableName,isTemporary
bronze,flights,false


database,tableName,isTemporary
silver,flights,false


database,tableName,isTemporary
gold,biweekly_distance,false
gold,hourly_flight_counts,false
gold,monthly_delays,false
gold,monthly_max_delay,false
gold,route_sliding_avg_delay,false
gold,status_sliding_counts,false
gold,weekly_route_distance,false


In [0]:
bronze_df = spark.table(f"`{catalog_name}`.bronze.flights")
bronze_df.display()
bronze_df.printSchema()
print(bronze_df.count())

date,delay,distance,origin,destination,_rescued_data,ingestion_timestamp
1011245,6,602,ABE,ATL,null,2026-03-28 20:35:05
1020600,-8,369,ABE,DTW,null,2026-03-28 20:35:05
1021245,-2,602,ABE,ATL,null,2026-03-28 20:35:05
1020605,-4,602,ABE,ATL,null,2026-03-28 20:35:05
1031245,-4,602,ABE,ATL,null,2026-03-28 20:35:05
1030605,0,602,ABE,ATL,null,2026-03-28 20:35:05
1041243,10,602,ABE,ATL,null,2026-03-28 20:35:05
1040605,28,602,ABE,ATL,null,2026-03-28 20:35:05
1051245,88,602,ABE,ATL,null,2026-03-28 20:35:05
1050605,9,602,ABE,ATL,null,2026-03-28 20:35:05


root
 |-- date: integer (nullable = true)
 |-- delay: integer (nullable = true)
 |-- distance: integer (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)

1391578


In [0]:
from pyspark.sql.functions import col

bronze_df.filter(col("ingestion_timestamp").isNull()).count()

0

In [0]:
silver_df = spark.table(f"`{catalog_name}`.silver.flights")
silver_df.display()
silver_df.printSchema()
print(silver_df.count())

date,departure,arrival,standardized_route,distance,delay,take_off_status,ingestion_timestamp,event_timestamp,seq
1011245,"Allentown, PA","Atlanta, GA",ABE-ATL,602,6,On Time,2026-03-28 20:35:05,2025-01-01T12:45:00.000Z,3
1020600,"Allentown, PA","Detroit, MI",ABE-DTW,369,-8,Early,2026-03-28 20:35:05,2025-01-02T06:00:00.000Z,3
1021245,"Allentown, PA","Atlanta, GA",ABE-ATL,602,-2,Early,2026-03-28 20:35:05,2025-01-02T12:45:00.000Z,3
1020605,"Allentown, PA","Atlanta, GA",ABE-ATL,602,-4,Early,2026-03-28 20:35:05,2025-01-02T06:05:00.000Z,3
1031245,"Allentown, PA","Atlanta, GA",ABE-ATL,602,-4,Early,2026-03-28 20:35:05,2025-01-03T12:45:00.000Z,3
1030605,"Allentown, PA","Atlanta, GA",ABE-ATL,602,0,On Time,2026-03-28 20:35:05,2025-01-03T06:05:00.000Z,3
1041243,"Allentown, PA","Atlanta, GA",ABE-ATL,602,10,Late,2026-03-28 20:35:05,2025-01-04T12:43:00.000Z,3
1040605,"Allentown, PA","Atlanta, GA",ABE-ATL,602,28,Late,2026-03-28 20:35:05,2025-01-04T06:05:00.000Z,3
1051245,"Allentown, PA","Atlanta, GA",ABE-ATL,602,88,Delay,2026-03-28 20:35:05,2025-01-05T12:45:00.000Z,3
1050605,"Allentown, PA","Atlanta, GA",ABE-ATL,602,9,On Time,2026-03-28 20:35:05,2025-01-05T06:05:00.000Z,3


root
 |-- date: string (nullable = true)
 |-- departure: string (nullable = true)
 |-- arrival: string (nullable = true)
 |-- standardized_route: string (nullable = true)
 |-- distance: integer (nullable = true)
 |-- delay: integer (nullable = true)
 |-- take_off_status: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- seq: integer (nullable = true)

1361141


In [0]:
silver_df.columns

['date',
 'departure',
 'arrival',
 'standardized_route',
 'distance',
 'delay',
 'take_off_status',
 'ingestion_timestamp',
 'event_timestamp',
 'seq']

In [0]:
from pyspark.sql.functions import col

print("Null event_timestamp:", silver_df.filter(col("event_timestamp").isNull()).count())
print("Empty departure:", silver_df.filter((col("departure").isNull()) | (col("departure") == "")).count())
print("Empty arrival:", silver_df.filter((col("arrival").isNull()) | (col("arrival") == "")).count())
print("Bad seq:", silver_df.filter(col("seq") != 3).count())

Null event_timestamp: 0
Empty departure: 0
Empty arrival: 0
Bad seq: 0


In [0]:
silver_df.groupBy("take_off_status").count().display()

take_off_status,count
On Time,336192
Early,651328
Late,181969
Delay,191652


In [0]:
silver_df.select("standardized_route").distinct().display()

standardized_route
ABE-ATL
ABE-DTW
ABE-ORD
ABI-DFW
ABQ-DFW
ABQ-JFK
ABQ-ATL
ABQ-MSP
ABQ-SLC
ABQ-IAH


In [0]:
spark.table(f"`{catalog_name}`.gold.monthly_delays").display()
spark.table(f"`{catalog_name}`.gold.monthly_delays").printSchema()

window_start,window_end,moving_avg_delay
2025-03-01 00:00:00,2025-04-01 00:00:00,0.16
2025-01-01 00:00:00,2025-02-01 00:00:00,0.25
2025-02-01 00:00:00,2025-03-01 00:00:00,0.2


root
 |-- window_start: string (nullable = true)
 |-- window_end: string (nullable = true)
 |-- moving_avg_delay: double (nullable = true)



In [0]:
spark.table(f"`{catalog_name}`.gold.biweekly_distance").display()

window_start,window_end,total_distance
2025-01-16 00:00:00,2025-01-30 00:00:00,137124712
2024-12-19 00:00:00,2025-01-02 00:00:00,9967392
2025-01-02 00:00:00,2025-01-16 00:00:00,143764431
2025-03-27 00:00:00,2025-04-10 00:00:00,54776385
2025-02-13 00:00:00,2025-02-27 00:00:00,145578616
2025-01-30 00:00:00,2025-02-13 00:00:00,137128874
2025-02-27 00:00:00,2025-03-13 00:00:00,149938128
2025-03-13 00:00:00,2025-03-27 00:00:00,154083895


In [0]:
spark.table(f"`{catalog_name}`.gold.monthly_max_delay").display()

window_start,window_end,worst_route,max_delay
2025-03-01 00:00:00,2025-04-01 00:00:00,DFW-TPA,27.37
2025-02-01 00:00:00,2025-03-01 00:00:00,ORD-SFO,27.3
2025-01-01 00:00:00,2025-02-01 00:00:00,BNA-DFW,26.0


In [0]:
spark.table(f"`{catalog_name}`.gold.hourly_flight_counts").display()

window_start,window_end,flight_count
2025-02-24 07:00:00,2025-02-24 08:00:00,1105
2025-02-19 10:00:00,2025-02-19 11:00:00,996
2025-02-09 05:00:00,2025-02-09 06:00:00,110
2025-01-14 11:00:00,2025-01-14 12:00:00,971
2025-02-05 20:00:00,2025-02-05 21:00:00,590
2025-01-13 07:00:00,2025-01-13 08:00:00,1085
2025-03-03 16:00:00,2025-03-03 17:00:00,963
2025-02-06 12:00:00,2025-02-06 13:00:00,967
2025-02-03 20:00:00,2025-02-03 21:00:00,614
2025-02-22 12:00:00,2025-02-22 13:00:00,860


In [0]:
for q in spark.streams.active:
    print("Name:", q.name)
    print("Id:", q.id)
    print("Status:", q.status)
    print("Recent progress:", q.recentProgress)
    print("-" * 80)

In [0]:
print("Bronze:", spark.table(f"`{catalog_name}`.bronze.flights").count())
print("Silver:", spark.table(f"`{catalog_name}`.silver.flights").count())
print("Monthly delays:", spark.table(f"`{catalog_name}`.gold.monthly_delays").count())
print("Biweekly distance:", spark.table(f"`{catalog_name}`.gold.biweekly_distance").count())
print("Monthly max delay:", spark.table(f"`{catalog_name}`.gold.monthly_max_delay").count())
print("Hourly counts:", spark.table(f"`{catalog_name}`.gold.hourly_flight_counts").count())

Bronze: 1391578
Silver: 1361141
Monthly delays: 3
Biweekly distance: 8
Monthly max delay: 3
Hourly counts: 1979


In [0]:
spark.table(f"`{catalog_name}`.silver.flights") \
    .filter(col("event_timestamp").isNull()) \
    .display()

date,departure,arrival,standardized_route,distance,delay,take_off_status,ingestion_timestamp,event_timestamp,seq
